# DiagnosticIQ source step 1: rule document extraction

Parses the expert rule document `$ASSETOPSBENCH_DATASET_DIR/data/skyspark_rules.pdf` into `extracted/extracted_data.json`.

This was the legacy first step in the source repository, where it was marked *DO NOT RUN*: running it again overwrites `extracted_data.json`, which every later step reads.

In [ ]:
import os
import fitz
import re
import json
from datetime import datetime
from typing import Optional, List, Dict

In [ ]:
#parameter 

from dataset_utils.paths import get_dataset_dir

dataset_dir = get_dataset_dir()
data_loc = dataset_dir / "data"
file_loc = os.path.join(data_loc, "skyspark_rules.pdf")
save_loc = os.path.join(dataset_dir, "extracted", "extracted_data.json")
os.makedirs(os.path.dirname(save_loc), exist_ok=True)

In [ ]:
def text_write(text, filename= "debug.test"):
    with open(filename, "w") as f0:
        f0.write(text)

def regex_inbetween(text, s1,s2) -> Optional[str]:
    start = text.find(s1)
    end = text.find(s2)

    if start == -1 or end == -1:
        return None
    
    return text[start+len(s1):end]

def rule_format(rule_str:str, rule_no)->dict:

    #print(rule_str)
    data = {}
    data["#n"] = rule_no
    data["id"] = rule_str.split("\n")[0]
    data["name"] = regex_inbetween(rule_str, 'Name \n', ' \nDisable')
    data["priority"] = None
    if data["name"] and data["name"].find("Priorit\ny \n") != -1:
        split_txt = data["name"].split("Priorit\ny \n")
        data["name"] = split_txt[0]
        data["priority"] = split_txt[1]

    data["disabled"] = regex_inbetween(rule_str, 'Disable\nd \n', ' \nFunctio\nn')
    data["function"] = regex_inbetween(rule_str, 'Functio\nn \n', ' \nDisplay \nText')
    data["display_text"] = regex_inbetween(rule_str, 'Display \nText \n', ' \nSource')
    data["code"] = regex_inbetween(rule_str, 'Source \n', ' \nend')
    
    if data["code"]:
        data["code"] += " \nend"

    return data

    

def format_regex(full_text:str) -> List[Dict]:
    rule_blocks = [x.strip() for x in full_text.strip().split("\nRule \n")]
    
    rule_set = []

    for i,rule in enumerate(rule_blocks):
        ext_rule = rule_format(rule, i)
        rule_set.append(ext_rule)

    return rule_set
        

def extract_dates(text:str) -> List[str]:
    pattern = r'\b\d{2}/\d{2}/\d{4}\b'
    dates = re.findall(pattern, text)
    return dates

def extract_lines(text:str) -> List[str]:
    pattern = r'\b\d{2}/\d{2}/\d{4}\b'
    dates = re.findall(pattern, text)
    return dates

def display_text_formatting(rule:str)->dict:
    data = {
        "text":rule,
        "header":None,
        "observations":None,
        "rules":None,
        "date":None,
        "saved":None
    }

    rule = "^*^*^\n"+rule+"\n^*^*^"

    dates = extract_dates(rule)
    if len(dates) == 1:
        data["date"] = dates[0]
        data["header"] = regex_inbetween(rule, "^*^*^\n", dates[0]).strip() if regex_inbetween(rule, "^*^*^\n", dates[0]) is not None else None

    if "This rule has been shown" in rule:
        data["saved"] = [x for x in rule.split("\n") if "This rule has been shown" in x]
        if data["saved"]:
            data["saved"] = data["saved"][0]

    if rule.find("What to look for: \n") != -1:
        start = rule.find("What to look for: \n")
        start += len("What to look for: \n")

        end = rule[start:].find(' \n \n')
        
        if end == -1:
            end = rule[start:].find('\n^*^*^') 
        
        end += start

        data["observations"] = rule[start:end].strip()

    if data["date"]:
        start = rule.find(data["date"])
        start += len(data["date"])

        if data["observations"]:
            end = rule[start:].find(' \n \n')

        else:
            if data["saved"]:
                end = rule[start:].find('This rule has been shown')
            else:
                end = rule[start:].find('\n^*^*^')

        end += start
        data["rules"] = rule[start:end].strip()

    return data

def display_text_cleaning(rule:dict) -> dict:
    if rule["rules"] != None:
        rule["rules"] = [x.strip() for x in rule["rules"].split(" - ") if not x.strip() == ""]
        rule["rules"][0] = rule["rules"][0][2:]

    if rule["observations"] != None:
        rule["observations"] = [x.strip() for x in rule["observations"].split(" - ") if not x.strip() == ""]
    
    return rule

def save_json(data:dict, loc:str) -> None:
    with open(loc, "w") as f0:
        json.dump(data, f0)

def read_file(loc:str):
    doc = fitz.open(loc)
    full_text = ""
    for page_no, page in enumerate(doc):
        text = page.get_text("text")
        full_text += text

    full_text = "\n".join(full_text.split("\n")[2:])
    rule_set = format_regex(full_text)

    for r in rule_set:
        r["display_text"] = display_text_formatting(r["display_text"])
        #r["display_text"] = display_text_cleaning(r["display_text"])


    data = {
        "date": datetime.now().isoformat(),
        "rule_set":rule_set
    }
    save_json(data, save_loc) 


In [ ]:
read_file(file_loc)

In [ ]:
with open(save_loc, "r") as f0:
    dataset = json.load(f0)

dataset = dataset["rule_set"]